In [22]:
import os
import requests
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
#from langchain_classic.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

In [10]:
llm=ChatOllama(model='llama3.1:8b')

In [12]:
question="Who is the present Prime minister of India?"

In [14]:
response=llm.invoke(question)
print(response)

content='As of my last update in 2023, Narendra Modi has been serving as the Prime Minister of India. He took office on May 26, 2014, and was re-elected for a second term in 2019. However, please note that political leadership positions can change over time due to elections or other events. For the most current information, I recommend checking with reliable news sources.' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-01-30T13:48:07.3565472Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4183924300, 'load_duration': 120721500, 'prompt_eval_count': 19, 'prompt_eval_duration': 51979900, 'eval_count': 81, 'eval_duration': 3982689300, 'logprobs': None, 'model_name': 'llama3.1:8b'} id='run--019c0f29-2ee3-7961-acd3-6c0f477404a2-0' usage_metadata={'input_tokens': 19, 'output_tokens': 81, 'total_tokens': 100}


In [16]:
pdf=PyPDFLoader('RAGPaper.pdf')
documents=pdf.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(documents)

In [20]:
embeddings=OllamaEmbeddings(model='llama3.1:8b')
db=FAISS.from_documents(documents=chunks,embedding=embeddings)

In [34]:
CONDENSE_QUESTION_PROMPT=PromptTemplate.from_template("""Given the following converstaion and a follow question,rephrase the follow up question to be a standalone question.
                                                       Chat History:{chat_history}
                                                       Follow Up Input:{question}
                                                       Standalone question:""")
qa=ConversationalRetrievalChain.from_llm(llm=llm,retriever=db.as_retriever(),condense_question_prompt=CONDENSE_QUESTION_PROMPT,return_source_documents=True,
                                        verbose=False)

In [36]:
chat_history=[]
query="""What is this research paper all about,explain it in 2-3 lines"""
result=qa({"question":query,"chat_history":chat_history})
print(result['answer'])

This research paper presents the RAG (Retrieval-Augmented Generator) model, which combines a language generator with a retriever to improve text generation and classification tasks. The model retrieves relevant information from external sources and incorporates it into its generation process, achieving competitive results on various benchmarks.
